# التكميم: ما تكلّفه الدقة الثمانية والرباعية فعلاً

**منحنى المقايضة ليس خطاً، وفيه هاوية** · معالج رسوميات · ~40 دقيقة · Colab

التكميم أرخص مكسب في النشر: خزّن الأوزان ببتّات أقل، فيتّسع النموذج على عتاد لم يكن يسعه من قبل. تخبرك الأوراق أنه ينجح. وما لا تستطيع إخبارك به هو ما يخسره نموذجك أنت، لأن الجواب يتوقف على النموذج والبيانات وموضع إنفاق الدقة في الشبكة.

فالسؤال المهم ليس «هل تنجح الدقة الرباعية» بل كيف يبدو المنحنى بين الدقة النصفية والرباعية. اقرأ أغلب الملخّصات تتوقّع انحداراً لطيفاً. قِسه تجد ما يشبه هضبةً تتلوها حافة، والحافة ليست حيث ينتصف عدد البتّات.

### الهدف

قِس الحيرة والذاكرة وزمن الاستجابة للنموذج ذاته بدقة نصفية وثمانية ورباعية، وارسم الثلاثة بعضها في مواجهة بعض، وحدِّد أيُّ التكاليف الثلاثة يتحرك أولاً وأيُّها لا يكاد يتحرك.

### الأوراق وراء هذه الورشة

- [llm-int8](https://azimuth.plus/ar/paper/llm-int8) — لماذا تنهار الدقة الثمانية الساذجة في النماذج الكبيرة، والبصيرة الشاذة التي تصلحها — ديتمرز وزملاؤه، 2022
- [gptq](https://azimuth.plus/ar/paper/gptq) — تكميم رباعي بخطوة واحدة يصمد، بتصغير الخطأ طبقةً طبقة — فرانتار وزملاؤه، 2023

> احفظ نسخة في Drive قبل أن تبدأ (ملف ← حفظ نسخة في Drive). التعديلات على الأصل لا تُحفظ.

## الإعداد

`PROFILE` هو المقبض الوحيد للحجم. المستوى المجاني هو الافتراضي ويعمل داخل حدود Colab المجانية.

In [ ]:
SLUG = "quantization-what-it-costs"
LANG = "ar"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import os
import subprocess
import sys
from pathlib import Path

# Guarded three ways. Colab users re-run the setup cell constantly, and
# a contributor may already be sitting inside a checkout — the first
# Windows run of this notebook cloned the repository into its own
# generated/notebooks/ directory because neither case was handled.
REPO = 'azimuth-workshops'
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'shim' / 'azimuth_nb').is_dir()), None)
if root is None:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    root = (here / REPO).resolve()
print('workshop root:', root)

# Absolute, so nothing depends on the working directory. Dropping the
# %cd this cell used to do also means the notebook stops caring where
# it was opened from.
sys.path.insert(0, str(root / 'shim'))
_ = os.environ.setdefault('AZIMUTH_DATA_DIR', str(root / 'data'))

In [ ]:
# Declared by this workshop (dependencies: in workshop.yaml).
# torch is NOT installed here — it is asserted, because a second
# torch over Colab's own will not match the driver.
%pip install -q transformers==4.44.2 bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0

ثلاثة أرقام تتحرك حين تُكمِّم، وهي لا تتحرك معاً. تنخفض الذاكرة تبعاً لعدد البتّات تقريباً، وهذا جزء حسابي. وتنخفض الجودة أيضاً لكن لا بسلاسة — تكاد لا تتحرك حيناً ثم تهوي. أما زمن الاستجابة فهو ما يفاجئ الناس، لأنه قد يسير في الاتجاه الخاطئ تماماً.

وقياس الثلاثة معاً هو المقصد. الورقة التي تذكر الحيرة وحدها لا تخفي شيئاً؛ إنها تجيب عن سؤال غير الذي بين يديك وأنت تقرّر ما تنشره.

_فحص مسبق. هذه أول ورشة هنا تحتاج معالج رسوميات فعلاً — نوى الدقة الثمانية والرباعية على CUDA وحدها._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

_ابنِ نوافذ التقييم مرة واحدة قبل تحميل أي نموذج. كل صيغة تُقيَّم على النص ذاته — فعيّنة مختلفة لكل نموذج تخفي تراجعاً حقيقياً داخل ضجيج المعاينة._

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL = env.cfg["model"]
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# BUILD THE WINDOWS ONCE, BEFORE ANY MODEL LOADS.
#
# Every variant is scored on byte-identical text. Re-sampling per model would
# put sampling noise on the same axis as the effect being measured, and a real
# int4 regression could hide inside it — the numbers would still look precise.
raw = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join(t for t in raw["text"] if t.strip())
all_ids = tokenizer(text, return_tensors="pt").input_ids[0][: env.cfg["evalTokens"]]

WINDOW = 512
windows = [all_ids[i : i + WINDOW] for i in range(0, len(all_ids) - WINDOW, WINDOW)]
n_windows = len(windows)

if env.lang == "ar":
    print(f"النموذج: {MODEL}")
    print(f"نوافذ التقييم: {n_windows} نافذة × {WINDOW} رمز")
else:
    print(f"model: {MODEL}")
    print(f"eval windows: {n_windows} × {WINDOW} tokens")

> **الورقة** · [llm-int8](https://azimuth.plus/ar/paper/llm-int8) — لماذا تنهار الدقة الثمانية الساذجة في النماذج الكبيرة، والبصيرة الشاذة التي تصلحها — ديتمرز وزملاؤه، 2022
>
> وجد ديتمرز وزملاؤه أن الدقة الثمانية الساذجة تنهار في النماذج الكبيرة لا لأن ثماني بتّات قليلة في المتوسط، بل لأن حفنة من أبعاد السمات الشاذة لها مدى لا تملكه البقية. والإبقاء على تلك بدقة أعلى هو الحل كله، ولهذا تكاد الدقة الثمانية أدناه تكون مجانية بينما لن تكون كذلك في تنفيذ ساذج.

_ثلاث عمليات تحميل، وثلاثة قياسات لكل منها. يُحرَّر النموذج بين الصيغ ليكون رقم الذاكرة للنموذج لا للبقايا._

In [ ]:
import gc
import time

from transformers import AutoModelForCausalLM, BitsAndBytesConfig


def load(variant):
    """One model, three ways. Only the dtype/quantization config differs."""
    if variant == "fp16":
        kwargs = {"torch_dtype": torch.float16}
    elif variant == "int8":
        kwargs = {"quantization_config": BitsAndBytesConfig(load_in_8bit=True)}
    else:
        kwargs = {
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                # NF4 rather than plain int4: GPTQ's lesson is that WHERE the
                # levels sit matters as much as how many there are.
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
        }
    return AutoModelForCausalLM.from_pretrained(MODEL, device_map="cuda:0", **kwargs)


def perplexity(model):
    """Mean NLL over the shared windows, exponentiated."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for window in windows:
            ids = window.unsqueeze(0).to("cuda:0")
            loss = model(ids, labels=ids).loss
            total += loss.item() * ids.numel()
            count += ids.numel()
    return float(torch.exp(torch.tensor(total / count)))


def footprint(model):
    """Megabytes of parameters as actually stored, not as declared.

    Reading nelement * element_size per parameter is the honest measure: a
    4-bit tensor reports element_size 1 with two values packed per byte, so
    counting declared dtypes would overstate int4 by exactly the factor the
    workshop is trying to measure.
    """
    return sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024**2


def latency(model, runs, new_tokens):
    """Median milliseconds per generated token, single sequence."""
    prompt = windows[0][:64].unsqueeze(0).to("cuda:0")
    with torch.no_grad():  # warm the kernels before timing anything
        model.generate(prompt, max_new_tokens=8, do_sample=False)
    torch.cuda.synchronize()
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        with torch.no_grad():
            model.generate(prompt, max_new_tokens=new_tokens, do_sample=False)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000 / new_tokens)
    times.sort()
    return times[len(times) // 2]


results = {}
for variant in ("fp16", "int8", "int4"):
    model = load(variant)
    results[variant] = {
        "ppl": perplexity(model),
        "mb": footprint(model),
        "ms": latency(model, env.cfg["latencyRuns"], env.cfg["newTokens"]),
    }
    print(
        f"  {variant:5} ppl {results[variant]['ppl']:7.3f}"
        f"  {results[variant]['mb']:8.1f} MB"
        f"  {results[variant]['ms']:6.1f} ms/token"
    )
    # Freed between variants so the memory figure is the model, not leftovers.
    del model
    gc.collect()
    torch.cuda.empty_cache()

fp16_ppl, int8_ppl, int4_ppl = (results[v]["ppl"] for v in ("fp16", "int8", "int4"))
fp16_mb, int8_mb, int4_mb = (results[v]["mb"] for v in ("fp16", "int8", "int4"))

> **الحجم** — يستخدم الملف المجاني {{scale.model}} و{{scale.evalTokens}} رمزاً من نص التقييم — صغير بما يكفي لتحميله ثلاث مرات على T4 دون إعادة اتصال. وشكل المنحنى هو ما ينتقل إلى نموذج أكبر، لا قيمة الحيرة المطلقة.

_ثلاث تكاليف في شكل واحد. اقرأ أي خط مستوٍ وأيّها فيه انعطاف — الانعطاف هو القرار، وهو ليس حيث ينتصف عدد البتّات._

In [ ]:
import matplotlib.pyplot as plt

memory_ratio = fp16_mb / int4_mb
ppl_ratio_int8 = int8_ppl / fp16_ppl
ppl_ratio_int4 = int4_ppl / fp16_ppl

variants = ["fp16", "int8", "int4"]
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, key, title in zip(
    axes,
    ["ppl", "mb", "ms"],
    ["perplexity", "memory (MB)", "ms / token"],
):
    values = [results[v][key] for v in variants]
    ax.plot(variants, values, marker="o", color="#457b9d")
    ax.set_title(title, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    # Zero-based so a flat line LOOKS flat. Autoscaled axes turn a 1% change
    # into a dramatic slope, which is how a chart lies without a wrong number
    # anywhere in it.
    ax.set_ylim(0, max(values) * 1.2)
fig.tight_layout()
plt.show()

if env.lang == "ar":
    print(f"الذاكرة: النصفية أكبر بـ {memory_ratio:.2f}× من الرباعية")
    print(f"الحيرة: ثمانية {ppl_ratio_int8:.3f}× · رباعية {ppl_ratio_int4:.3f}×")
else:
    print(f"memory: fp16 is {memory_ratio:.2f}× larger than int4")
    print(f"perplexity: int8 {ppl_ratio_int8:.3f}× · int4 {ppl_ratio_int4:.3f}×")

_الرقم الذي يسير في الاتجاه الخاطئ. أوزان أصغر وعمل أكثر في كل ضرب مصفوفات._

In [ ]:
fp16_ms, int8_ms, int4_ms = (results[v]["ms"] for v in ("fp16", "int8", "int4"))

# Report the DIRECTION, because the direction is the surprise. Quantized
# weights are smaller but every matmul now pays a dequantize step, so at batch
# size 1 the smaller model is often the slower one.
if env.lang == "ar":
    print(
        f"زمن الاستجابة: نصفية {fp16_ms:.1f} · ثمانية {int8_ms:.1f} · رباعية {int4_ms:.1f} م.ث/رمز"
    )
    verdict = "أبطأ" if int4_ms > fp16_ms else "أسرع"
    print(
        f"الرباعية {verdict} من النصفية بعامل {max(int4_ms, fp16_ms) / min(int4_ms, fp16_ms):.2f}"
    )
else:
    print(f"latency: fp16 {fp16_ms:.1f} · int8 {int8_ms:.1f} · int4 {int4_ms:.1f} ms/token")
    verdict = "SLOWER" if int4_ms > fp16_ms else "faster"
    print(f"int4 is {verdict} than fp16 by {max(int4_ms, fp16_ms) / min(int4_ms, fp16_ms):.2f}×")

### تمرين — pick-a-deployment

اختر صيغة لنشر محدَّد، ودافع عنها بأرقامك أنت لا بالافتراضية. اكتب القيد أولاً — «يتّسع في 8 غيغابايت»، «أقل من 50 مللي ثانية لكل رمز»، «جودة في حدود 2٪ من النصفية» — ثم اقرأ الجدول واختر.

ثم لاحظ ما فعله بك التمرين. إن أُعطيت رقماً واحداً حسّنته، وإن أُعطيت ثلاثة لزمك تحديد قيد، والقيد قرار منتج لا قرار تقني. أيُّ الثلاثة لن تساوم عليه أبداً، وهل يتغير جوابك لروبوت محادثة مقابل ملخِّص دفعات؟

_تلميح متاح: `env.hint(2)`_

In [ ]:
# YOUR TURN.
#
# State the constraint BEFORE reading the table. Then let the table choose.
BUDGET_MB = 400
MAX_MS_PER_TOKEN = 60
MAX_PPL_RATIO = 1.05

viable = [
    v
    for v in variants
    if results[v]["mb"] <= BUDGET_MB
    and results[v]["ms"] <= MAX_MS_PER_TOKEN
    and results[v]["ppl"] / fp16_ppl <= MAX_PPL_RATIO
]
print(f"viable under your constraints: {viable or 'none — loosen one, and say which'}")

_على الذاكرة أن تنخفض فعلاً، وعلى الدقة الثمانية أن تصمد فعلاً. وإن تحركت الثمانية كثيراً فالتلميح الثالث هو السبب._

In [ ]:
memory_ok = env.check("memory-falls", memory_ratio)
quality_ok = env.check("quality-survives-int8", ppl_ratio_int8)

In [ ]:
receipt = env.receipt()